In [ ]:
import os
os.chdir("path")

In [ ]:
from src.api import asteroid_data_extract, rec_to_df, fetch_dates_asteroids, fetch_daterange_asteroids
from src.processing import standardise_asteroid_data, asteroid_data_report, validate_report
from src.database import insert_events, get_dates, get_period_dates

In [ ]:
#checking table 'close_approaches' exists
import sqlite3
c = sqlite3.connect("data/nasa_asteroids.db")
cursor = c.cursor()
cursor.execute("""
    SELECT name
    FROM sqlite_master
    WHERE type = 'table';
""")

print(cursor.fetchall())
c.close()

In [ ]:
#testing insertion for one date 2020-01-01, if report is okay insert
asteroids = fetch_dates_asteroids("2020-01-01")
rec = asteroid_data_extract(asteroids)
df = rec_to_df(rec)
sdf = standardise_asteroid_data(df)
report = asteroid_data_report(sdf)
print(df)

In [ ]:
insert_events(sdf)

In [ ]:
import sqlite3
#use this to check number of rows
with sqlite3.connect("data/nasa_asteroids.db") as connection:

    cursor = connection.cursor()

    cursor.execute("""
        SELECT COUNT(*)
        FROM close_approaches;
    """)

    print(cursor.fetchone())

In [ ]:
#testing api extraction and database insertion on month of january 2020
start_date, end_date = "2020-01-01", "2020-01-31"
jan2020df = fetch_daterange_asteroids(start_date, end_date)
jan2020sdf = standardise_asteroid_data(jan2020df)
j2020report = asteroid_data_report(jan2020sdf)
print(report)

In [ ]:
#passed report so insert
insert_events(jan2020sdf)

### API to SQL
Below is the pipeline, I am taking in data in 6 month batches and inserting it into the SQLite database. Although potentially made automatic, as I will be running into API rate limits I'm handling manually.

In [ ]:
#pipeline, edit dates for entry
start_year = 2020
start_month = 10
end_year = 2020
end_month = 12
#up to dec 2020 complete

batch_dates = get_period_dates(start_month, start_year, end_month, end_year)

In [ ]:
#inserts events from the above period into SQLite database
for start_date, end_date in batch_dates:
    month_df = fetch_daterange_asteroids(start_date, end_date)
    month_sdf = standardise_asteroid_data(month_df)
    month_report = asteroid_data_report(month_sdf)
    validate_report(month_report)
    insert_events(month_sdf)